# Predictive Multiplicity Experiment Guide

This notebook inspects the COMPAS data, compares model classes on XOR, withholds a feature in a controlled synthetic example, and measures what happens when a recorded feature is included in COMPAS. Accuracy is measured on the sample used in each experiment.

### 1. Locate the data files

**What this cell does and why**

List files under `data` relative to the current working directory to check the data path before loading the CSV.

In [2]:
from pathlib import Path

for path in sorted(Path("data").rglob("*")):
    if path.is_file():
        print(path)

**How to interpret the result**

An empty output means no files were found under `data` relative to the current working directory. The next cell uses `../data` relative to the notebook directory.

### 2. Load the COMPAS data

**What this cell does and why**

Read the CSV and display its dimensions, column names, and first five rows to verify that the required features and label are available.

In [3]:
import pandas as pd

df = pd.read_csv("../data/compas_arrest_processed.csv")

print("num_rows, num_columns:", df.shape)
print("columns:", df.columns.tolist())
display(df.head())

num_rows, num_columns: (6172, 22)
columns: ['arrest', 'race_is_causasian', 'race_is_african_american', 'race_is_hispanic', 'race_is_other', 'age_leq_25', 'age_25_to_45', 'age_geq_46', 'female', 'n_priors_eq_0', 'n_priors_geq_1', 'n_priors_geq_2', 'n_priors_geq_5', 'n_juvenile_misdemeanors_eq_0', 'n_juvenile_misdemeanors_geq_1', 'n_juvenile_misdemeanors_geq_2', 'n_juvenile_misdemeanors_geq_5', 'n_juvenile_felonies_eq_0', 'n_juvenile_felonies_geq_1', 'n_juvenile_felonies_geq_2', 'n_juvenile_felonies_geq_5', 'charge_degree_eq_M']


,arrest,race_is_causasian,race_is_african_american,race_is_hispanic,race_is_other,age_leq_25,age_25_to_45,age_geq_46,female,n_priors_eq_0,...,n_priors_geq_5,n_juvenile_misdemeanors_eq_0,n_juvenile_misdemeanors_geq_1,n_juvenile_misdemeanors_geq_2,n_juvenile_misdemeanors_geq_5,n_juvenile_felonies_eq_0,n_juvenile_felonies_geq_1,n_juvenile_felonies_geq_2,n_juvenile_felonies_geq_5,charge_degree_eq_M
0,-1,0,0,0,1,0,0,1,0,1,...,0,1,0,0,0,1,0,0,0,0
1,1,0,1,0,0,0,1,0,0,1,...,0,1,0,0,0,1,0,0,0,0
2,1,0,1,0,0,1,0,0,0,0,...,0,1,0,0,0,1,0,0,0,0
3,-1,0,0,0,1,0,1,0,0,1,...,0,1,0,0,0,1,0,0,0,1
4,1,1,0,0,0,0,1,0,0,0,...,1,1,0,0,0,1,0,0,0,0


**How to interpret the result**

The sample has 6,172 rows and 22 columns. The target label is `arrest`.

### 3. Inspect values by column

**What this cell does and why**

Print up to five distinct nonmissing values per column to inspect how the features and labels are encoded.

In [ ]:
for col in df.columns:
    examples = df[col].dropna().unique()[:5].tolist()
    print(f"{col} → {examples}")

**How to interpret the result**

The selected features use 0 and 1, while the label uses −1 and +1. This output shows example values. The next step counts their frequencies for the selected inputs.

### 4. Count COMPAS labels by feature combination

**What this cell does and why**

Use age at most 25 and at least one juvenile felony as `x1` and `x2`. Count labels in each input group to see whether identical inputs can have different labels.

In [5]:
data = df[["age_leq_25", "n_juvenile_felonies_geq_1", "arrest"]].copy()
data.columns = ["x1", "x2", "y"]

groups = pd.MultiIndex.from_product([[0, 1], [0, 1]], names=["x1", "x2"])
counts = pd.crosstab([data["x1"], data["x2"]], data["y"])
counts = counts.reindex(index=groups, columns=[-1, 1], fill_value=0)
# counts["total_cnt_rows"] = counts.sum(axis=1)

display(counts)

y        -1     1
x1 x2            
0  0   2728  1953
   1     42   102
1  0    576   707
   1     17    47

**How to interpret the result**

For example, `(0, 0)` occurs with label −1 in 2,728 cases and with label +1 in 1,953 cases. A model using these two features must give every case in that group the same prediction, so some errors are unavoidable.

### 5. Construct the XOR dataset

**What this cell does and why**

Assign label 1 when the two feature values differ and label 0 when they match. This gives us fixed inputs and labels for comparing the expressiveness of the two model classes.

In [6]:
import pandas as pd
from itertools import product

# The four binary inputs and their XOR labels
xor_data = pd.DataFrame({
    "x1": [0, 0, 1, 1],
    "x2": [0, 1, 0, 1],
    "y":  [0, 1, 1, 0],
})

# Count labels 0 and 1 for each input combination.
xor_counts = pd.crosstab([xor_data["x1"], xor_data["x2"]], xor_data["y"])
display(xor_counts)

y      0  1
x1 x2      
0  0   1  0
   1   0  1
1  0   0  1
   1   1  0

**How to interpret the result**

The two inputs labeled 1 occupy opposite corners of the square. A single straight line cannot separate them from the two inputs labeled 0.

### 6. Score the possible prediction patterns

**What this cell does and why**

Enumerate and score all 16 binary prediction patterns on the four inputs using the label counts. Trees of depth at most two can represent all 16. Linear classifiers can represent 14 because the two diagonal patterns are not linearly separable.

In [ ]:
# Score linear and tree prediction patterns using the label counts.
def score_models(label_counts, negative, positive):
    # Set the order of the four inputs in each prediction pattern.
    groups = [(0, 0), (0, 1), (1, 0), (1, 1)]
    # Count the total number of cases in the label table.
    total = int(label_counts[[negative, positive]].to_numpy().sum())

    # Two diagonal patterns that a linear classifier cannot represent
    nonlinear_patterns = {
        (negative, positive, positive, negative),
        (positive, negative, negative, positive),
    }

    results = []

    for predictions in product([negative, positive], repeat=4):
        # Add the number of matching labels in each group.
        correct = 0
        for group, prediction in zip(groups, predictions):
            correct += int(label_counts.at[group, prediction])

        model_classes = ["Decision tree (depth ≤ 2)"]
        if predictions not in nonlinear_patterns:
            model_classes.append("Linear classifier")

        for model_class in model_classes:
            results.append({
                "Model class": model_class,
                "Predictions (①, ②, ③, ④)": predictions,
                "Correct": correct,
                "Incorrect": total - correct,
            })

    return pd.DataFrame(results)


xor_results = score_models(xor_counts, negative=0, positive=1)

**How to interpret the result**

Scoring every candidate lets us identify all patterns tied for the best score in each class.

### 7. Compare optimal patterns on XOR

**What this cell does and why**

Keep every pattern that attains the highest score in its model class. Compare both the best score and the number of patterns tied at that score.

In [ ]:
# Keep all patterns tied for the best score in each model class.
best_score = xor_results.groupby("Model class")["Correct"].transform("max")
xor_best = xor_results[xor_results["Correct"] == best_score]

display(xor_best.sort_values("Model class").reset_index(drop=True))

,Model class,"Predictions (①, ②, ③, ④)",Correct,Incorrect
0,Decision tree (depth ≤ 2),"(0, 1, 1, 0)",4,0
1,Linear classifier,"(0, 0, 1, 0)",3,1
2,Linear classifier,"(0, 1, 0, 0)",3,1
3,Linear classifier,"(0, 1, 1, 1)",3,1
4,Linear classifier,"(1, 1, 1, 0)",3,1


**How to interpret the result**

Four linear patterns each classify 3 of 4 cases correctly. One tree pattern classifies all 4 correctly.

### 8. Apply the model comparison to COMPAS

**What this cell does and why**

Pass the COMPAS label counts for the two selected features to the same scoring function to see whether the result resembles XOR.

In [ ]:
# Score both model classes against the recorded COMPAS labels.
compas_results = score_models(counts, negative=-1, positive=1)

best_score = compas_results.groupby("Model class")["Correct"].transform("max")
compas_best = compas_results[compas_results["Correct"] == best_score]

display(compas_best.reset_index(drop=True))

,Model class,"Predictions (①, ②, ③, ④)",Correct,Incorrect
0,Decision tree (depth ≤ 2),"(-1, 1, 1, 1)",3584,2588
1,Linear classifier,"(-1, 1, 1, 1)",3584,2588


**How to interpret the result**

Both classes have one optimal pattern, `(-1, 1, 1, 1)`, which correctly classifies 3,584 cases (58.07%). Each input group has a unique majority label, so errors remain even though the optimal prediction pattern is unique.

### 9. Compare linear predictions against a reference

**What this cell does and why**

Place the four optimal XOR patterns in rows and the four inputs in columns. Mark a cell `True` if its prediction differs from reference pattern A.

In [ ]:
# The four optimal linear prediction patterns
patterns = xor_best.loc[
    xor_best["Model class"] == "Linear classifier",
    "Predictions (①, ②, ③, ④)"
]
print(patterns)

# Each row is a prediction pattern. Each column is an input.
answers = pd.DataFrame(
    patterns.tolist(),
    columns=["① (0,0)", "② (0,1)", "③ (1,0)", "④ (1,1)"]
)
answers.index = ["Reference A", "B", "C", "D"]
display(answers)

# Mark entries that differ from reference pattern A.
# The ne method returns True where the values differ.
different = answers.ne(answers.loc["Reference A"], axis="columns")
display(different)

5     (0, 0, 1, 0)
9     (0, 1, 0, 0)
14    (0, 1, 1, 1)
27    (1, 1, 1, 0)
Name: Predictions (①, ②, ③, ④), dtype: object


,"① (0,0)","② (0,1)","③ (1,0)","④ (1,1)"
Reference A,0,0,1,0
B,0,1,0,0
C,0,1,1,1
D,1,1,1,0


,"① (0,0)","② (0,1)","③ (1,0)","④ (1,1)"
Reference A,False,False,False,False
B,False,True,True,False
C,False,True,False,True
D,True,True,False,False


**How to interpret the result**

For example, B differs from A on `(0, 1)` and `(1, 0)`. The next cell summarizes this table with two disagreement measures.

### 10. Measure disagreement among linear classifiers

**What this cell does and why**

Compute ambiguity as the fraction of cases where at least one candidate disagrees with the reference. Compute discrepancy as the largest fraction of cases on which a single candidate disagrees with the reference.

In [12]:
# Check whether any pattern disagrees on each input.
changed_by_question = different.any(axis=0)
display(changed_by_question)

# Count the disagreements with A for each pattern.
changes_by_model = different.sum(axis=1)
display(changes_by_model)

n_questions = different.shape[1]  # Four XOR inputs
ambiguity = changed_by_question.sum() / n_questions
discrepancy = changes_by_model.max() / n_questions

print(f"Ambiguity: {ambiguity:.0%}")
print(f"Discrepancy: {discrepancy:.0%}")

① (0,0)    True
② (0,1)    True
③ (1,0)    True
④ (1,1)    True
dtype: bool

Reference A    0
B       2
C       2
D       2
dtype: int64

Ambiguity: 100%
Discrepancy: 50%


**How to interpret the result**

Ambiguity of 100% means that each of the four cases has at least one optimal pattern that disagrees with the reference. Discrepancy of 50% means the candidate that differs most from the reference disagrees on two of the four cases.

### 11. Compare tree predictions against a reference

**What this cell does and why**

Take the optimal tree patterns and compare them with the first pattern, using the same procedure as for the linear classifiers.

In [13]:
tree_patterns = xor_best.loc[
    xor_best["Model class"] == "Decision tree (depth ≤ 2)",
    "Predictions (①, ②, ③, ④)"
]

tree_answers = pd.DataFrame(
    tree_patterns.tolist(),
    columns=["① (0,0)", "② (0,1)", "③ (1,0)", "④ (1,1)"]
)

# Use the first optimal prediction pattern as the reference.
tree_A = tree_answers.iloc[0]
tree_different = tree_answers.ne(tree_A, axis="columns")

display(tree_answers)
display(tree_different)

,"① (0,0)","② (0,1)","③ (1,0)","④ (1,1)"
0,0,1,1,0


,"① (0,0)","② (0,1)","③ (1,0)","④ (1,1)"
0,False,False,False,False


**How to interpret the result**

Every cell is `False` because there is only one optimal prediction pattern.

### 12. Measure disagreement among trees

**What this cell does and why**

Apply the same formulas to the tree predictions on the four XOR cases so the two model classes can be compared directly.

In [14]:
# Check whether any pattern disagrees with the reference on each input.
changed_by_question = tree_different.any(axis=0)

# Count each pattern’s disagreements with the reference.
changes_by_model = tree_different.sum(axis=1)

n_questions = tree_different.shape[1]  # Four inputs
ambiguity = changed_by_question.sum() / n_questions
discrepancy = changes_by_model.max() / n_questions

print("Any disagreement by input?", changed_by_question.tolist())
print("Disagreements per pattern:", changes_by_model.tolist())
print(f"Ambiguity: {ambiguity:.0%}")
print(f"Discrepancy: {discrepancy:.0%}")

Any disagreement by input? [False, False, False, False]
Disagreements per pattern: [0]
Ambiguity: 0%
Discrepancy: 0%


**How to interpret the result**

Both ambiguity and discrepancy are 0%. The unique optimal prediction pattern correctly classifies all four cases.

## Prediction disagreement from a missing feature

Set the true label to `y = x3`. Use the same five cases and labels in both conditions, and vary only which features are available to trees of depth at most two. Count identical prediction vectors once. Both disagreement measures use all five cases as their denominator.

### 13. Withhold the feature that determines the synthetic labels

**What this cell does and why**

Assign `y = x3` to each of five cases. The `tree_predictions` function combines possible splits and leaf predictions to enumerate patterns from trees of depth at most two. Keep the labels and maximum depth fixed and vary access to `x3`.

In [1]:
from itertools import product

# Each row is (x1, x2, x3), with x3 used directly as the label.
feature_rows = [(0, 0, 0), (0, 0, 1), (0, 1, 1), (1, 0, 1), (1, 1, 0)]
labels = tuple(row[2] for row in feature_rows)


def tree_predictions(rows, feature_indices, depth):
    # A leaf predicts the same label for all rows it receives.
    patterns = {(label,) * len(rows) for label in (0, 1)}
    if depth == 0:
        return patterns

    for feature in feature_indices:
        left = [row for row in rows if row[feature] == 0]
        right = [row for row in rows if row[feature] == 1]
        if not left or not right:
            continue
        left_patterns = tree_predictions(left, feature_indices, depth - 1)
        right_patterns = tree_predictions(right, feature_indices, depth - 1)
        for left_pattern, right_pattern in product(left_patterns, right_patterns):
            predictions = (iter(left_pattern), iter(right_pattern))
            patterns.add(tuple(next(predictions[row[feature]]) for row in rows))
    return patterns


for condition, features in [("x3 withheld", (0, 1)), ("x3 included", (0, 1, 2))]:
    patterns = tree_predictions(feature_rows, features, depth=2)
    scores = {
        pattern: sum(prediction == label for prediction, label in zip(pattern, labels))
        for pattern in patterns
    }
    best_score = max(scores.values())
    best_patterns = sorted(pattern for pattern, score in scores.items() if score == best_score)
    reference = best_patterns[0]
    ambiguity = sum(len(set(column)) > 1 for column in zip(*best_patterns)) / len(labels)
    discrepancy = max(
        sum(prediction != baseline for prediction, baseline in zip(pattern, reference))
        for pattern in best_patterns
    ) / len(labels)

    print(f"{condition} | Best accuracy: {best_score}/{len(labels)}")
    print(f"Optimal patterns ({len(best_patterns)}): {best_patterns}")
    print(f"Ambiguity: {ambiguity:.0%} | Discrepancy: {discrepancy:.0%}")

x3 withheld | Best accuracy: 4/5
Optimal patterns (2): [(0, 0, 1, 1, 0), (1, 1, 1, 1, 0)]
Ambiguity: 40% | Discrepancy: 40%
x3 included | Best accuracy: 5/5
Optimal patterns (1): [(0, 1, 1, 1, 0)]
Ambiguity: 0% | Discrepancy: 0%


**How to interpret the result**

With `x3` withheld, the first two cases have identical observed inputs. Predicting 0 for both or 1 for both gives two patterns tied at 4/5, with ambiguity and discrepancy both at 40% (2/5). With `x3` included, the unique optimal pattern scores 5/5, and both measures fall to 0%.

## Withholding and restoring a feature in COMPAS

Add `n_priors_geq_1`, an indicator for at least one prior offense, to the original two features. Compare all distinct patterns from binary decision trees of depth at most two on the same records and observed labels. The reference in each condition is the lexicographically first optimal prediction pattern. Ambiguity counts cases where any candidate differs from the reference. Discrepancy measures the fraction of cases on which the single most divergent candidate differs from the reference. Each group is weighted by its observed frequency.

This comparison uses the recorded COMPAS labels as they are. The preceding synthetic experiment isolates feature omission under a known, error-free labeling rule.

### 14. Compare COMPAS models with and without the prior-offense feature

**What this cell does and why**

Use the same 6,172 COMPAS cases, observed labels, and tree depth limit of two in both conditions. Vary whether the prior-offense feature is available. For each condition, keep patterns within 0, 0.5, 1, and 2 percentage points of its best accuracy. Weight each input group by its number of cases.

In [2]:
import csv
from collections import Counter
from pathlib import Path

compas_features = ("age_leq_25", "n_juvenile_felonies_geq_1", "n_priors_geq_1")
compas_path = Path("../data/compas_arrest_processed.csv")
if not compas_path.is_file():
    compas_path = Path("data/compas_arrest_processed.csv")

with compas_path.open(newline="") as file:
    compas_rows = [
        (tuple(int(row[feature]) for feature in compas_features), int(row["arrest"]) == 1)
        for row in csv.DictReader(file)
    ]

sample_size = len(compas_rows)
print("Condition | Best accuracy | Tolerance (pp) | Pattern count | Ambiguity | Discrepancy")
for condition, feature_count in [("Prior-offense feature withheld", 2), ("Prior-offense feature included", 3)]:
    label_counts = Counter((row[:feature_count], label) for row, label in compas_rows)
    groups = sorted({row for row, _ in label_counts})
    weights = [label_counts[row, 0] + label_counts[row, 1] for row in groups]
    patterns = tree_predictions(groups, range(feature_count), depth=2)
    correct_counts = {
        pattern: sum(label_counts[row, prediction] for row, prediction in zip(groups, pattern))
        for pattern in patterns
    }
    best_correct = max(correct_counts.values())
    reference = min(pattern for pattern, correct in correct_counts.items() if correct == best_correct)

    for tolerance_bp in (0, 50, 100, 200):
        near_optimal = [
            pattern for pattern, correct in correct_counts.items()
            if (best_correct - correct) * 10000 <= tolerance_bp * sample_size
        ]
        ambiguity = sum(
            weight for weight, predictions in zip(weights, zip(*near_optimal))
            if len(set(predictions)) > 1
        ) / sample_size
        discrepancy = max(
            sum(weight for weight, a, b in zip(weights, pattern, reference) if a != b)
            for pattern in near_optimal
        ) / sample_size
        print(
            f"{condition} | {best_correct / sample_size:.2%} | {tolerance_bp / 100:g} | "
            f"{len(near_optimal)} | {ambiguity:.2%} | {discrepancy:.2%}"
        )

Condition | Best accuracy | Tolerance (pp) | Pattern count | Ambiguity | Discrepancy
Prior-offense feature withheld | 58.07% | 0 | 1 | 0.00% | 0.00%
Prior-offense feature withheld | 58.07% | 0.5 | 2 | 1.04% | 1.04%
Prior-offense feature withheld | 58.07% | 1 | 3 | 3.37% | 2.33%
Prior-offense feature withheld | 58.07% | 2 | 4 | 3.37% | 3.37%
Prior-offense feature included | 59.95% | 0 | 1 | 0.00% | 0.00%
Prior-offense feature included | 59.95% | 0.5 | 3 | 53.09% | 52.06%
Prior-offense feature included | 59.95% | 1 | 4 | 55.43% | 52.06%
Prior-offense feature included | 59.95% | 2 | 9 | 65.42% | 62.05%


**How to interpret the result**

A tolerance of 0 keeps only exactly optimal patterns. At 0.5 percentage points, candidates may score up to 0.5 percentage points below their own condition’s optimum. With the prior-offense feature included, ambiguity at this tolerance rises from 1.04% to 53.09%. Each condition has one exactly optimal pattern. These are in-sample results using the observed COMPAS labels.

### Interpreting the COMPAS results

| Condition | Best in-sample accuracy | Exactly optimal patterns | Patterns within 0.5 pp of the optimum | Ambiguity within 0.5 pp |
|---|---:|---:|---:|---:|
| Prior-offense feature withheld | 58.07% | 1 | 2 | 1.04% |
| Prior-offense feature included | 59.95% | 1 | 3 | 53.09% |

With the prior-offense feature included, both the best in-sample accuracy and disagreement among near-optimal patterns increased. The group with both original features set to 0 and the prior-offense feature set to 1 contains 3,213 cases. Of these, 1,621 have label `+1` and 1,592 have label `-1`. Flipping this group's prediction changes 3,213 predictions and reduces the number of correct predictions by only 29, or about 0.47 percentage points.

Each condition has one exactly optimal prediction pattern. These results describe one feature comparison on the observed COMPAS sample. The earlier synthetic experiment shows how withholding a feature can leave multiple exactly optimal patterns when the labeling rule is known.